# Big Sur Marathon

The course in plan and section, drawn in three line weights from USGS elevation. Run `fetch_week1.py day2` first.

## 1 · Settings

In [ ]:
DAY      = 2
PROMPT   = "Lines"
SLUG     = "lines-big-sur"
NOTEBOOK = "day02_lines_big_sur.ipynb"
TITLE    = "Big Sur\nMarathon"
CREDITS  = "Course facts: Big Sur International Marathon. Route: © OpenStreetMap contributors. Elevation and contours: USGS 3DEP."

## 2 · Load

In [ ]:
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from scipy import ndimage
from mapkit import poster, publish
from mapkit.paths import PROCESSED

route = pd.read_csv(PROCESSED / "bigsur_profile.csv")
boston = pd.read_csv(PROCESSED / "boston_comm_ave.csv")
dem_file = np.load(PROCESSED / "bigsur_dem.npz")
dem = dem_file["dem_dm"] / 10.0
x0, y0, x1, y1 = dem_file["bbox_3857"]

## 3 · Analysis, every number on the map comes from here

In [ ]:
def flatten_bridges(df):
    """Elevation models read the canyon under a bridge, so we draw a straight deck across each one."""
    z = df.elev_ft.copy()
    z[df.bridge] = np.nan
    return z.interpolate(limit_direction="both")

route["z"] = flatten_bridges(route)
boston["z"] = boston.elev_ft.rolling(5, center=True, min_periods=1).mean()   # smooth out street level noise

def biggest_climb(df):
    """The largest rise from any low point to a later high point."""
    low = df.z.cummin()
    top = (df.z - low).idxmax()
    bottom = df.z[:top + 1].idxmin()
    return bottom, top, float(df.z[top] - df.z[bottom])

# Hurricane Point: from the Little Sur River to the top of the course
a = route[(route.dist_mi >= 9.5) & (route.dist_mi <= 12.5)]
hp_low, hp_top, hp_rise = biggest_climb(a)
hp_miles = route.dist_mi[hp_top] - route.dist_mi[hp_low]
# Heartbreak Hill: the last big climb on Commonwealth Avenue
b = boston[(boston.dist_mi >= 1.0) & (boston.dist_mi <= 2.0)]    # past the earlier Newton hills
h_low, h_top, h_rise = biggest_climb(b)
h_miles = boston.dist_mi[h_top] - boston.dist_mi[h_low]
ratio = hp_rise / h_rise
print(f"Route length       {route.dist_mi.max():.2f} mi  (official 26.2)")
print(f"Top of the course  {route.z.max():.0f} ft  (official 560)")
print(f"Hurricane Point    {hp_rise:.0f} ft over {hp_miles:.2f} mi, mile {route.dist_mi[hp_low]:.1f} to {route.dist_mi[hp_top]:.1f}  (official chart: 40 to 560 ft = 520)")
print(f"Heartbreak Hill    {h_rise:.0f} ft over {h_miles:.2f} mi  (published 88 to 91)")
print(f"Ratio              {ratio:.1f}x")
print(f"Total climbing     {route.z.diff().clip(lower=0).sum():.0f} ft  (published 2,182)")

## 4 · Map

In [ ]:
PAPER, INK, RED = "#f3efe6", "#1f1e1b", "#d4380d"
fig, ax = poster.page(PAPER)
fx0, fy0, fx1, fy1 = poster.fit_frame(ax, -122.06, 36.20, -121.68, 36.58)

# clean up bad pixels, and keep only the mainland
bad = dem > 1800
if bad.any():
    dem[bad] = ndimage.median_filter(dem, size=9)[bad]
labels, _ = ndimage.label(ndimage.binary_opening(dem > 1, iterations=2))
sizes = np.bincount(labels.ravel())
sizes[0] = 0
is_land = labels == sizes.argmax()
smooth = ndimage.gaussian_filter(np.where(is_land, dem, 0), 2.0)
gx = np.linspace(x0, x1, dem.shape[1])
gy = np.linspace(y1, y0, dem.shape[0])

# Plan: three line weights. Hairlines every 20 m, medium every 100 m, heavy every 500 m.
for step, weight, alpha in [(20, 0.14, 0.55), (100, 0.42, 0.85), (500, 1.15, 1.0)]:
    ax.contour(gx, gy, smooth, levels=np.arange(step, 1800, step), colors=INK, linewidths=weight, alpha=alpha, zorder=2)

# The ocean: lines leaving the shore, spreading out and thinning as they go
to_shore = ndimage.distance_transform_edt(~is_land) * (x1 - x0) / dem.shape[1]
for i, d in enumerate([120, 300, 560, 900, 1350, 1950, 2750, 3800, 5200]):
    ax.contour(gx, gy, to_shore, levels=[d], colors=INK, linewidths=max(0.14, 0.75 - i * 0.08), alpha=max(0.2, 0.9 - i * 0.09), zorder=1)
ax.contour(gx, gy, is_land.astype(float), levels=[0.5], colors=INK, linewidths=1.6, zorder=3)
ax.set_xlim(fx0, fx1); ax.set_ylim(fy0, fy1)

# The course in one color, heavier on the big climb
rx, ry = poster.merc(route.lon, route.lat)
ax.plot(rx, ry, color=PAPER, lw=5, solid_capstyle="round", zorder=5)
ax.plot(rx, ry, color=RED, lw=1.7, solid_capstyle="round", zorder=6)
climb = route.loc[hp_low:hp_top]
cx, cy = poster.merc(climb.lon, climb.lat)
ax.plot(cx, cy, color=RED, lw=4.6, solid_capstyle="round", zorder=7)

def mark(mile_at, text, side="right", color=INK):
    i = (route.dist_mi - mile_at).abs().idxmin()
    mx, my = poster.merc(route.lon[i], route.lat[i])
    ax.scatter([mx], [my], s=30, color="white", edgecolor=RED, lw=1, zorder=9)
    gap = (fx1 - fx0) * 0.014
    poster.halo_text(ax, mx + (gap if side == "right" else -gap), my, text, color=color, size=12.5, path=poster.LATO_BOLD, halo=PAPER,
                     va="center", zorder=10, ha="left" if side == "right" else "right")

mark(0, "Big Sur")
mark(11, "Hurricane Point", color=RED)
mark(route.dist_mi.max(), "Carmel", side="left")

# Section: the course cut open along its length, small and quiet in the open ocean
sec = fig.add_axes([0.06, 0.098, 0.4, 0.085], facecolor="none")

def hatch(df, base, color, start, every=0.13, weight=0.24):
    """Fine vertical lines under a profile."""
    miles = np.arange(df.dist_mi.min(), df.dist_mi.max(), every)
    heights = np.interp(miles, df.dist_mi, df.z - base)
    sec.vlines(miles - df.dist_mi.min() + start, 0, heights, color=color, lw=weight)

hatch(route, 0, INK, 0)
sec.plot(route.dist_mi, route.z, color=INK, lw=0.75)
hatch(climb, 0, RED, route.dist_mi[hp_low], every=0.05, weight=0.42)
sec.plot(climb.dist_mi, climb.z, color=RED, lw=1.6, solid_capstyle="round")
sec.plot([0, route.dist_mi.max()], [0, 0], color=INK, lw=0.6)

# Heartbreak Hill, to the same scale on both axes
hb = boston.loc[h_low:h_top].copy()
hb_at = 29.5
hatch(hb, boston.z[h_low], RED, hb_at, every=0.05, weight=0.42)
sec.plot(hb_at + hb.dist_mi - hb.dist_mi.iloc[0], hb.z - boston.z[h_low], color=RED, lw=1.6, solid_capstyle="round")
sec.plot([28.9, 30.7], [0, 0], color=INK, lw=0.6)
sec.set_xlim(0, 31); sec.set_ylim(0, 640)
sec.set_axis_off()
tiny = poster.font(poster.LATO, 8.8)
sec.text(0, -35, "0", color=INK, fontproperties=tiny, ha="left", va="top", clip_on=False)
sec.text(route.dist_mi.max(), -35, "26.2 miles", color=INK, fontproperties=tiny, ha="right", va="top", clip_on=False)
sec.text(route.dist_mi[hp_top] + 0.45, route.z[hp_top] - 20, f"Hurricane Point, {hp_rise:.0f} ft climb", color=RED, fontproperties=poster.font(poster.LATO_BOLD, 9.5), va="center")
sec.text(hb_at + 0.3, -35, f"Heartbreak Hill,\n{h_rise:.0f} ft", color=RED, fontproperties=poster.font(poster.LATO_BOLD, 8.8), ha="center", va="top", linespacing=1.2, clip_on=False)

poster.title(fig, TITLE, x=0.06, y=0.72, size=46, color=INK, path=poster.OLD, sub_path=poster.LATO, sub_color="#4b4943", sub_size=13.5,
             subtitle="26.2 miles along California's\nHighway 1. The world's largest\nrural marathon.")
poster.credit(fig, CREDITS, color="#66635a", path=poster.LATO, name_path=poster.LATO_BOLD, name_color=INK, size=8.5)

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle="The course in plan and section, drawn in three line weights", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)